# high_confidence_subset

Create conservative and relaxed candidate subsets before building the first baseline.

Idea:
- prefer higher-rated recordings
- downweight recordings with many secondary labels
- inspect class loss after filtering
- compare a strict rule against a relaxed rule
- rescue classes where rating metadata is unusable

In [ ]:
from pathlib import Path
import ast

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

ROOT = Path.cwd().resolve()
if ROOT.name == "exp000_eda":
    ROOT = ROOT.parents[1]
elif not (ROOT / "train.csv").exists():
    raise FileNotFoundError("Run this notebook from the repository root or experiments/exp000_eda.")

ROOT

In [ ]:
train = pd.read_csv(ROOT / "train.csv")
taxonomy = pd.read_csv(ROOT / "taxonomy.csv")

train["secondary_count"] = (
    train["secondary_labels"]
    .fillna("[]")
    .map(lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0)
)

train["rating"] = pd.to_numeric(train["rating"], errors="coerce")
train[["primary_label", "rating", "secondary_count", "type"]].head()

## Filter definitions

In [ ]:
STRICT_MIN_RATING = 4.0
STRICT_MAX_SECONDARY = 1

RELAXED_MIN_RATING = 3.5
RELAXED_MAX_SECONDARY = 2


def build_subset(df: pd.DataFrame, min_rating: float, max_secondary: int) -> pd.DataFrame:
    subset = df.loc[df["rating"].fillna(0) >= min_rating]
    subset = subset.loc[subset["secondary_count"] <= max_secondary].copy()
    return subset


strict = build_subset(train, STRICT_MIN_RATING, STRICT_MAX_SECONDARY)
relaxed = build_subset(train, RELAXED_MIN_RATING, RELAXED_MAX_SECONDARY)

## Rating-less class rescue

If a class has no usable rating metadata at all, do not let the rating threshold delete it completely.

In [ ]:
rating_profile = (
    train.groupby("primary_label")
    .agg(
        n=("filename", "size"),
        rating_max=("rating", "max"),
        rating_mean=("rating", "mean"),
        secondary_mean=("secondary_count", "mean"),
    )
)

ratingless_labels = rating_profile.index[rating_profile["rating_max"].fillna(0) <= 0].tolist()
len(ratingless_labels), ratingless_labels[:20]

In [ ]:
rescued_unrated = train.loc[
    train["primary_label"].isin(ratingless_labels)
].copy()

rescued = pd.concat([strict, rescued_unrated], ignore_index=True).drop_duplicates(subset=["filename"])
rescued

In [ ]:
comparison = pd.DataFrame([
    {
        "subset": "strict",
        "rows": len(strict),
        "retention": len(strict) / len(train),
        "classes": strict["primary_label"].nunique(),
    },
    {
        "subset": "relaxed",
        "rows": len(relaxed),
        "retention": len(relaxed) / len(train),
        "classes": relaxed["primary_label"].nunique(),
    },
    {
        "subset": "strict_plus_ratingless",
        "rows": len(rescued),
        "retention": len(rescued) / len(train),
        "classes": rescued["primary_label"].nunique(),
    },
])
comparison

In [ ]:
orig_counts = train["primary_label"].value_counts().rename("original_count")
strict_counts = strict["primary_label"].value_counts().rename("strict_count")
relaxed_counts = relaxed["primary_label"].value_counts().rename("relaxed_count")
rescued_counts = rescued["primary_label"].value_counts().rename("rescued_count")

coverage = pd.concat([orig_counts, strict_counts, relaxed_counts, rescued_counts], axis=1).fillna(0)
coverage[["strict_count", "relaxed_count", "rescued_count"]] = coverage[["strict_count", "relaxed_count", "rescued_count"]].astype(int)
coverage["strict_retention"] = coverage["strict_count"] / coverage["original_count"]
coverage["relaxed_retention"] = coverage["relaxed_count"] / coverage["original_count"]
coverage["rescued_retention"] = coverage["rescued_count"] / coverage["original_count"]
coverage["rescued_by_relaxed"] = (coverage["strict_count"] == 0) & (coverage["relaxed_count"] > 0)
coverage["rescued_by_ratingless"] = (coverage["strict_count"] == 0) & (coverage["rescued_count"] > 0)
coverage.head()

In [ ]:
coverage.sort_values(["rescued_count", "original_count"], ascending=[False, False]).head(20)

In [ ]:
strict_lost = coverage.loc[coverage["strict_count"] == 0].sort_values("original_count", ascending=False)
strict_lost.head(20)

In [ ]:
rescued_relaxed = coverage.loc[coverage["rescued_by_relaxed"]].sort_values("relaxed_count", ascending=False)
rescued_relaxed.head(20)

In [ ]:
rescued_ratingless = coverage.loc[coverage["rescued_by_ratingless"]].sort_values("rescued_count", ascending=False)
rescued_ratingless.head(20)

In [ ]:
plot_df = comparison.copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=plot_df, x="subset", y="rows", ax=axes[0], hue="subset", palette=["#2d6a4f", "#3b7ddd", "#bc6c25"], legend=False)
axes[0].set_title("Rows retained")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=plot_df, x="subset", y="classes", ax=axes[1], hue="subset", palette=["#2d6a4f", "#3b7ddd", "#bc6c25"], legend=False)
axes[1].set_title("Classes retained")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()

## Investigate lost classes

Focus on the largest classes that disappear under the strict rule. These are the most likely candidates for class-specific exceptions.

In [ ]:
lost_labels = strict_lost.head(10).index.tolist()
lost_labels

In [ ]:
lost_profile = (
    train.loc[train["primary_label"].isin(lost_labels)]
    .groupby("primary_label")
    .agg(
        n=("filename", "size"),
        rating_mean=("rating", "mean"),
        rating_median=("rating", "median"),
        rating_max=("rating", "max"),
        secondary_mean=("secondary_count", "mean"),
        secondary_median=("secondary_count", "median"),
        unique_types=("type", lambda x: x.fillna("<missing>").nunique()),
    )
    .sort_values("n", ascending=False)
)
lost_profile

In [ ]:
focus_label = lost_labels[0] if lost_labels else None
focus_label

In [ ]:
if focus_label is not None:
    focus_rows = train.loc[train["primary_label"] == focus_label, ["filename", "rating", "secondary_count", "type", "secondary_labels"]].copy()
    display(focus_rows.head(20))
    display(focus_rows[["rating", "secondary_count"]].describe())

## Working decision for first submission

Use this project stage to get a submission out quickly rather than optimize filtering too deeply.

Recommended short-term plan:
- Train the first baseline on `strict_plus_ratingless`.
- Keep the full training set available as a later comparison.
- Do not over-engineer denoising yet.
- After the first submission, compare score before deciding whether to widen the training set further or add class-specific exceptions.

## Export candidate subsets

Keep these local unless you decide the format is stable enough to track.

In [ ]:
strict_path = ROOT / "experiments" / "exp000_eda" / "high_confidence_candidates.csv"
relaxed_path = ROOT / "experiments" / "exp000_eda" / "high_confidence_candidates_relaxed.csv"
rescued_path = ROOT / "experiments" / "exp000_eda" / "high_confidence_candidates_rescued.csv"

strict.to_csv(strict_path, index=False)
relaxed.to_csv(relaxed_path, index=False)
rescued.to_csv(rescued_path, index=False)

strict_path, relaxed_path, rescued_path

## Next questions

- How many classes are recovered by the rating-less rescue?
- Does the first submission work better with `strict_plus_ratingless` than strict alone?
- Should some rescued classes still be downweighted later?
- After the first score, is it better to widen data or improve features/modeling?